In [1]:
from pyspark.sql.functions import col, trim, upper, to_date

StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 3, Finished, Available, Finished, False)

In [2]:
df_customers_raw = spark.read.parquet('Files/customers/customers.parquet')
df_claims_raw = spark.read.parquet('Files/claims/claims.parquet')
df_policies_raw = spark.read.parquet('Files/policies/policies.parquet')

StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 4, Finished, Available, Finished, False)

In [3]:
display(df_customers_raw.limit(10))

StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, ba01c89b-4449-4746-bcf9-0109f04f0d44)

In [4]:
display(df_claims_raw.limit(10))

StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 560dde01-472d-4cc6-ba9c-ea4d98cc7a3d)

In [5]:
display(df_policies_raw.limit(10))

StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 540dc61a-82e6-47d7-8869-64670887ea99)

#### Data cleaning & data ingest into silver layer

In [6]:
## Customers transform
df_customers_raw = spark.read.parquet('Files/customers/customers.parquet')

df_customers_silver = df_customers_raw \
                .withColumn("cust_id", trim(col("cust_id")).cast('int'))\
                .withColumn("full_name", trim(col("Full Name")))\
                .withColumn("date_of_birth", to_date(col("Date_of_Birth")))\
                .withColumn("gender", upper(trim(col("Gender"))))\
                .withColumn("contact", trim(col("Contact")))\
                .select("cust_id", "full_name", "date_of_birth", "gender", "contact")\
                .dropDuplicates(["cust_id"])



StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 8, Finished, Available, Finished, False)

In [7]:
silver_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_silver.Lakehouse/Tables/dbo/dim_customers"

df_customers_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .save(silver_table_path)

StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 9, Finished, Available, Finished, False)

In [8]:
## polices transformation
df_policies_raw = spark.read.parquet('Files/policies/policies.parquet')


df_policies_silver = df_policies_raw \
    .withColumn("policy_id", trim(col("policy_id")).cast("int")) \
    .withColumn("cust_id", trim(col("cust_id")).cast("int")) \
    .withColumn("policy_type", upper(trim(col("policy_type")))) \
    .withColumn("status", upper(trim(col("status")))) \
    .withColumn("start_date", to_date(trim(col("Start_Date")), "dd/MM/yyyy")) \
    .withColumn("coverage_amount", col("Coverage_Amount").cast("decimal(18, 2)")) \
    .select("policy_id", "cust_id", "policy_type", "status", "start_date", "coverage_amount") \
    .dropDuplicates(["policy_id"])


StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 10, Finished, Available, Finished, False)

In [9]:
silver_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_silver.Lakehouse/Tables/dbo/dim_policies"

df_policies_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .save(silver_table_path)


StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 11, Finished, Available, Finished, False)

In [10]:
## claims transformation 

df_claims_raw = spark.read.parquet('Files/claims/claims.parquet')

df_claims_silver = df_claims_raw \
    .withColumn("claim_id", trim(col("claim_id")).cast("int")) \
    .withColumn("policy_id", trim(col("policy_id")).cast("int")) \
    .withColumn("claim_date", to_date(col("claim_date"))) \
    .withColumn("claim_amount", col("claim_amount").cast("decimal(18, 2)")) \
    .withColumn("claim_status", upper(trim(col("status")))) \
    .select("claim_id", "policy_id", "claim_date", "claim_amount", "claim_status") \
    .dropDuplicates(["claim_id"])


StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 12, Finished, Available, Finished, False)

In [11]:
silver_table_path = "abfss://insurance@onelake.dfs.fabric.microsoft.com/lh_insurance_silver.Lakehouse/Tables/dbo/fact_claims"

df_claims_silver.write\
    .format("delta")\
    .mode("overwrite")\
    .save(silver_table_path)



StatementMeta(, 5d72c0a1-3a0f-4e4c-a28c-62170b9e743d, 13, Finished, Available, Finished, False)